# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page, for one client, on one day (identified by
**content_id**, **client_id**, and **report_date** together). I will use the
**fact_content_daily_performance** table, filtered to **month=2026-03** —
a mid-panel month, not the final sealed month (June 2026, which is only
for testing query mechanics, not for label development).

In [19]:
!pip install -q duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

# install and load httpfs extension
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Set token through secret
con.sql(f"""
    CREATE SECRET hf_token_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
""")

print("Setup done!")

Setup done!


In [20]:
schema_check = con.sql("""
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    LIMIT 5
""").df()

print(schema_check.columns.tolist())
schema_check

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [21]:
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context** (IDs, grouping only — never features):
- report_date, client_hash_id, content_hash_id, month

**Feature** (knowable before the decision, safe to use):
- gsc_impressions, gsc_clicks, gsc_avg_position
- ga4_pageviews, ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec
- sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai
- scroll_events

**Label / proxy**:
- No pre-built decline/trend label exists in this table, so I will build a
  proxy: pages with high **gsc_impressions** but low CTR (clicks/impressions)
  are "underperforming", good visibility but not converting into clicks,
  which is a refresh opportunity.

**Excluded**:
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available
  — I'm only using these to check if the data is complete (in Query 3),
  not as model features, because they just tell us whether data exists,
  not how the content actually performed.
  
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude,
  ai_other. I already have sessions_ai which is the total of all these
  combined. Breaking it down by each individual AI tool adds too much detail
  that I don't need right now, so I'll keep it simple and just use the total.

In [22]:
avail_types = con.sql("""
    SELECT DISTINCT gsc_data_available, ga4_data_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    LIMIT 10
""").df()

avail_types


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,ga4_data_available
0,True,<NA>
1,False,<NA>
2,False,False
3,True,False
4,False,True
5,True,True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
counts = con.sql("""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

counts


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [24]:
availability = con.sql("""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

availability

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


**Query 1: Grain check:** To confirm one row really means one content
item, for one client, on one day, I grouped the data by report_date,
client_hash_id, and content_hash_id and checked if any group had more
than one row. It came back with 0 duplicate combinations, so the
grain holds.

**Query 2: Row count + date span:** My (month=2026-03) slice has **9841378**
total rows, covering dates from [2026-03-01] to [2026-03-31]. That lines up with what
I'd expect for one month of daily data.

**Query 3: Availability check:** Not every row has complete data, so I
checked this directly. Out of **9841378** total rows, only 3611061.0 have
gsc_data_available IS TRUE, and 413966.0 have ga4_data_available IS TRUE.
This is why I'm filtering on these availability flags before trusting any
GSC or GA4 signal, some clients or time periods just don't have full
coverage.

**Five features:**

1. gsc_impressions: available when? Known at the end of each day, as
   soon as GSC logs how often the page appeared in search — no future
   info needed.
2. gsc_avg_position: available when? Known at the end of each day,
   based on that day's search ranking data.
3. ga4_engaged_sessions: available when? Known at the end of each day,
   once GA4 finishes counting that day's sessions.
4. ga4_total_engagement_sec: available when? Known at the end of each
   day, from that day's tracked engagement time.
5. scroll_events: available when? Known at the end of each day, logged
   as users scroll through the page that day.

In [25]:
features_frame = con.sql("""
    SELECT
        report_date, client_hash_id, content_hash_id,
        gsc_impressions, gsc_avg_position,
        ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    LIMIT 10
""").df()

features_frame

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,ga4_total_engagement_sec,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,<NA>,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,7.347280,<NA>,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,7.832461,<NA>,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,3.272727,<NA>,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,5.636364,<NA>,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,4.500000,<NA>,<NA>,<NA>


In [26]:
df_leak = con.sql("""
    SELECT
        report_date, client_hash_id, content_hash_id,
        gsc_impressions, gsc_clicks, gsc_avg_position,
        ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_data_available IS TRUE
""").df()

df_leak.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3611061, 9)

In [27]:
# Label
import numpy as np

df_leak['ctr'] = np.where(df_leak['gsc_impressions'] > 0,
                            df_leak['gsc_clicks'] / df_leak['gsc_impressions'],
                            0)

df_leak['is_underperforming'] = (df_leak['ctr'] == 0).astype(int)
print(df_leak['is_underperforming'].value_counts())

is_underperforming
1    3193080
0     417981
Name: count, dtype: int64


In [28]:
# Honest score
from sklearn.tree import DecisionTreeClassifier

features_honest = ['gsc_avg_position', 'ga4_engaged_sessions',
                    'ga4_total_engagement_sec', 'scroll_events']

X_honest = df_leak[features_honest].fillna(0)
y = df_leak['is_underperforming']

tree_honest = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
tree_honest.fit(X_honest, y)

print("Honest accuracy:", round(tree_honest.score(X_honest, y), 3))

Honest accuracy: 0.561


In [29]:
# Leaky
features_leaky = features_honest + ['gsc_clicks']

X_leaky = df_leak[features_leaky].fillna(0)

tree_leaky = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
tree_leaky.fit(X_leaky, y)

print("Leaky accuracy:", round(tree_leaky.score(X_leaky, y), 3))

Leaky accuracy: 1.0


**The leakage trap:**

I built a proxy label (is_underperforming) based on whether ctr (clicks
÷ impressions) was zero. Using only honest, pre-decision features
(gsc_avg_position, ga4_engaged_sessions, ga4_total_engagement_sec,
scroll_events), my model scored **0.56** accuracy, a real but modest
signal.

Then I deliberately added gsc_clicks as a feature, the same column my
label is directly derived from. The accuracy jumped to **1.0**, which
looks amazing but is meaningless: the model isn't learning a pattern,
it's just reading the answer back from a feature that IS the label in
disguise.

I removed gsc_clicks from the feature set and kept the honest score of
**0.56** as the real, trustworthy result. This is the same leakage lesson
from notebook 02, performed here on real warehouse data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

There's a lot this data just can't tell me. A few things I need to keep
in mind:

- **Unbalanced history:** Clients don't all have the same amount of history.
  Some have years of GSC or GA4 data, others only a few months. So if I
  compare "declining" or "opportunity" scores across clients, it's not
  really a fair comparison, a client with less history might just look
  different because there's less data, not because their content is
  actually worse.

- **GSC-only early rows:** Some clients connected GA4 after they'd already
  been using GSC for a while. For those early rows, I only have GSC data,  
  the GA4 fields are empty and marked as unavailable. That doesn't mean
  engagement was zero on those days, it just means nobody was tracking it
  yet. I need to be careful not to read that as "no engagement."

- **Window overlaps:** For this notebook I'm only working with one month
  (month=2026-03), so I'm not combining overlapping windows right now.
  But if I bring in fields like (_last_30d) or (_prev_30d) later, those
  windows can overlap, and I'll need to line them up carefully so I don't
  end up double-counting the same days.

Given all this, I'd say what I'm building is decision-support, not a fully
fair, apples-to-apples comparison across every client and time period.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.